<a href="https://colab.research.google.com/github/jiminmini/mini/blob/ESAA_OB/9_8_%ED%95%84%EC%82%AC%EA%B3%BC%EC%A0%9C__%EC%B5%9C%EC%A2%85%EB%B3%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**[개념 정리]**

##**[투표기반 분류기]**

- 직접 투표: 직접튜표 분류기

##**[배깅과 페이스팅]**

- 배깅: 훈련 세트에서 중복 허용하여 샘플링 (반복)

- 페이스팅: 중복 허용하지 않고 샘플링

- 수집 함수: 분류일 때는 통계적 최빈값, 회귀는 평균 계산

###**[사이킷런의 배깅과 페이스팅]**

- 사이킷런: 간편한 API로 구성된 BaggingClassifier

- 일반적으로 배깅을 더 선호

##**[oob 평가]**

- oob_score=True로 지정> 훈련이 끝난 후 자동으로 oob 평가 수행

##**[엑스트라 트리]**

- 익스트릠 랜덤 트리: 극단적으로 무작위한 트리의 랜덤 포레스트

#**[코드 필사]**

In [1]:
import warnings
warnings.filterwarnings('ignore')

# import package
import numpy as np
import os

# 5장에서의 moons dataset 불러오기
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
X,y = make_moons(n_samples=100, noise=0.15)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [2]:
from sklearn.datasets import make_moons
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

x,y=make_moons(n_samples=100, noise=0.15)
polynomial_svm_clf=Pipeline([
    ("poly_features", PolynomialFeatures(degree=3)),
    ("scaler",StandardScaler()),
    ("svm_clf", LinearSVC(C=10, loss="hinge"))
])
polynomial_svm_clf.fit(x,y)

Pipeline(steps=[('poly_features', PolynomialFeatures(degree=3)),
                ('scaler', StandardScaler()),
                ('svm_clf', LinearSVC(C=10, loss='hinge'))])

In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42
)

log_clf=LogisticRegression()
rnd_clf=RandomForestClassifier()
svm_clf=SVC()


voting_clf=VotingClassifier(
    estimators=[('lr', log_clf), ('rf', rnd_clf), ('svc', svm_clf)],
    voting='hard')
voting_clf.fit(x_train,y_train)

VotingClassifier(estimators=[('lr', LogisticRegression()),
                             ('rf', RandomForestClassifier()), ('svc', SVC())])

In [4]:
from sklearn.metrics import accuracy_score
for clf in (log_clf, rnd_clf, svm_clf, voting_clf):
    clf.fit(x_train, y_train)
    y_pred=clf.predict(x_test)
    print(clf.__class__.__name__, accuracy_score(y_test, y_pred))

LogisticRegression 0.85
RandomForestClassifier 0.95
SVC 0.9
VotingClassifier 0.9


In [6]:
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier

bag_clf=BaggingClassifier(
    DecisionTreeClassifier(), n_estimators=500,
    max_samples=50, bootstrap=True, n_jobs=-1)
bag_clf.fit(x_train, y_train)
y_pred=bag_clf.predict(x_test)

In [7]:
bag_clf= BaggingClassifier(
    DecisionTreeClassifier(), n_estimators=500,
    bootstrap=True, n_jobs=-1, oob_score=True)
bag_clf.fit(x_train, y_train)
bag_clf.oob_score_

0.8875

In [8]:
from sklearn.metrics import accuracy_score
y_pred=bag_clf.predict(x_test)
accuracy_score(y_test, y_pred)

0.95

In [9]:
bag_clf.oob_decision_function_

array([[1.        , 0.        ],
       [0.01630435, 0.98369565],
       [1.        , 0.        ],
       [0.55494505, 0.44505495],
       [0.5       , 0.5       ],
       [0.        , 1.        ],
       [0.65921788, 0.34078212],
       [0.19526627, 0.80473373],
       [0.        , 1.        ],
       [0.62702703, 0.37297297],
       [0.04      , 0.96      ],
       [0.98857143, 0.01142857],
       [0.01570681, 0.98429319],
       [0.11363636, 0.88636364],
       [1.        , 0.        ],
       [1.        , 0.        ],
       [1.        , 0.        ],
       [0.        , 1.        ],
       [0.02105263, 0.97894737],
       [0.16751269, 0.83248731],
       [1.        , 0.        ],
       [0.87640449, 0.12359551],
       [0.04411765, 0.95588235],
       [0.04945055, 0.95054945],
       [0.80232558, 0.19767442],
       [0.27840909, 0.72159091],
       [0.99425287, 0.00574713],
       [0.47150259, 0.52849741],
       [0.91666667, 0.08333333],
       [0.00625   , 0.99375   ],
       [1.

In [10]:
from sklearn.ensemble import RandomForestClassifier

rnd_clf=RandomForestClassifier(n_estimators=500, max_leaf_nodes=16, n_jobs=-1)
rnd_clf.fit(x_train, y_train)

y_pred_rf=rnd_clf.predict(x_test)

In [11]:
bag_clf=BaggingClassifier(
    DecisionTreeClassifier(max_features="auto", max_leaf_nodes=16),
    n_estimators=500, max_samples=1.0, bootstrap=True, n_jobs=-1)

In [12]:
from sklearn.datasets import load_iris
iris=load_iris()
rnd_clf=RandomForestClassifier(n_estimators=500, n_jobs=-1)
rnd_clf.fit(iris["data"], iris["target"])
for name, score in zip(iris["feature_names"], rnd_clf.feature_importances_):
    print(name, score)

sepal length (cm) 0.09461107524036608
sepal width (cm) 0.023078143165693538
petal length (cm) 0.4287304979963919
petal width (cm) 0.4535802835975485
